In [1]:
import os
import sys
import pickle
import subprocess
import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display

sys.path.insert(0, os.path.abspath('../../../'))
from util.utils import ResourceMonitor, W2, generate_animation, generate_W2distance_plot


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
[KeOps] Warning : OpenMP library not found, it must be downloaded through Homebrew for apple Silicon chips
[KeOps] Warning : OpenMP support is not available. Disabling OpenMP.


In [ ]:
# parameters
dataset =  "stem_cell_differentiation"#"Axolotl_data_2000" "stem_cell_differentiation" "EMT_72" "LARRY_3000_benchmark"
d_red = 2
days = [0, 2, 4]
intermediate_days = [1, 3]

In [ ]:
# Load the preprocessed dataset from our Preprocess_datasets notebook

filename = f"../../../data/{dataset}_preprocessed.pkl"

with open(filename, "rb") as fr:
    time_label, full_matrix, projected_matrix, pca = pickle.load(fr)

# we primarily use (PCA-) projected_matrix unless we apply TrajectoryNet on the original datasets
projected_matrix = np.array(projected_matrix, dtype=np.float32)
time_label = np.array(time_label, dtype=np.float32)

In [ ]:
# Input variables for TrajectoryNet

npzname = f'../../../data/{dataset}.npz'

snapshots = []
snapshots_label = []

for i, day in enumerate(days):
    snapshots.append(projected_matrix[time_label==day, :d_red])
    snapshots_label.append(i * np.ones(snapshots[-1].shape[0]))  # 0-based index required by TrajectoryNet

embedding_matrix = np.concatenate(snapshots, axis=0)
sample_labels = np.concatenate(snapshots_label, axis=0)
print(embedding_matrix.shape, sample_labels.shape)

np.savez(npzname, pca=embedding_matrix, sample_labels=sample_labels)

In [ ]:
savedir = f"../../../assets/TrajectoryNet/{dataset}_dim{d_red}"

cmd = [
    "python3", "-m", "TrajectoryNet.main",
    "--dataset", npzname,
    "--save", savedir,
    "--embedding_name", "pca",
    "--niter", "10000",
    "--dims", "128-128-128",
    "--max_dim", str(d_red),
]
with ResourceMonitor() as monitor:
    subprocess.run(cmd, check=True)
monitor.report("TrajectoryNet training", filename=os.path.join(savedir, "resources.txt"))

In [ ]:
# Evaluate TrajectoryNet
!python3 -m TrajectoryNet.eval --dataset $npzname --save $savedir \
--embedding_name pca --dims "128-128-128" --max_dim $d_red

In [ ]:
savefile = f"{savedir}/backward_trajectories.npy"
traj = np.load(savefile)  # shape: (T, N, D)

X1_trpts = [traj[-i-1] for i in range(traj.shape[0])]
example_name = dataset
dimension_reduction = True


In [ ]:
savefile = f"{savedir}/trajectories.pkl"
# 저장
with open(savefile, "wb") as fw:
    pickle.dump(X1_trpts, fw)

print(f"✅ X1_trpts saved successfully to: {savefile}")

In [ ]:
img_src1 = f"{savedir}/particle_trajectories_.gif"
generate_animation(dataset, days, intermediate_days, X1_trpts, img_src1, d_red, dimension_reduction, plot_vectorfield=False)

img_src2 = f"{savedir}/velocity_trajectories_.gif"
generate_animation(dataset, days, intermediate_days, X1_trpts, img_src2, d_red, dimension_reduction, plot_vectorfield=True)

img_src3 = f"{savedir}/w2distances_.png"
generate_W2distance_plot(dataset, days, intermediate_days, X1_trpts, img_src3, d_red, dimension_reduction)


In [ ]:
for i, day in enumerate(days):
    plt.scatter(snapshots[i][:, 0], snapshots[i][:,1],  alpha=0.5, s=5, color = contrast_colors[day], label = f"day {day}")
plt.xlabel("x1")
plt.ylabel("x2")
plt.legend()
plt.title("True snapshots")

xlims = plt.xlim()
ylims = plt.ylim()
plt.show()


In [ ]:
idx_days = [int(day/dt/days[-1]) for day in days]
for i, day in zip(idx_days, days):
    plt.scatter(X1_trpts[i][:, 0], X1_trpts[i][:,1],  alpha=0.5, s=5, color = contrast_colors[day], label = f"day {day}")
plt.xlabel("x1")
plt.ylabel("x2")
plt.xlim(xlims)
plt.ylim(ylims)
plt.legend()
plt.title("Learned snapshots")
plt.show()


In [ ]:
# First plot: True snapshots
fig, axs = plt.subplots(1, 2, figsize=(8, 3))  # side-by-side layout

# Plot true snapshots
for i, day in enumerate(days):
    axs[0].scatter(snapshots[i][:, 0], snapshots[i][:, 1], alpha=0.5, s=5, color=contrast_colors[day], label=f"day {day}")
axs[0].set_xlabel("PC 1")
axs[0].set_ylabel("PC 2")
axs[0].set_title("True snapshots")
axs[0].legend()

# Capture x and y limits for synchronization
xlims = axs[0].get_xlim()
ylims = axs[0].get_ylim()

# Plot learned snapshots
idx_days = [int(day / dt / days[-1]) for day in days]
for i, day in zip(idx_days, days):
    axs[1].scatter(X1_trpts[i][:, 0], X1_trpts[i][:, 1], alpha=0.5, s=5, color=contrast_colors[day], label=f"day {day}")
axs[1].set_xlabel("PC 1")
axs[1].set_ylabel("PC 2")
axs[1].set_title("Learned snapshots")
axs[1].set_xlim(xlims)
axs[1].set_ylim(ylims)
axs[1].legend()

plt.tight_layout()
plt.savefig(f"{savedir}/result-{dataset}_dim{d_red}.png", dpi=300, bbox_inches='tight') 
plt.show()